# Baselines Inteligentes
### Naive, Seasonal Naive y Holt-Winters

`Fase 3 · Video 12` — serie **Forecasting de Ventas de la A a la Z**

Arranca la **Fase 3 (modelado)**. Regla de oro: un modelo solo es bueno si le gana a algo simple. Aquí
fijamos el **listón** —el Seasonal Naive— que todos los modelos siguientes deberán superar (MASE < 1).

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')

y = (df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum()
       .iloc[1:-1])
y.name = 'demanda'

print(f'✅ Serie semanal: {len(y)} semanas | {y.index.min().date()} → {y.index.max().date()}')

---
## 2. ¿Por qué necesitas un baseline?

Un **baseline** es un modelo trivial que sirve de vara de medir. Si tu modelo sofisticado no le gana, no
vale la complejidad, el costo de cómputo ni el riesgo de mantenimiento. Los tres clásicos:

| Baseline | Predicción | Captura |
|---|---|---|
| **Naive** | El último valor observado | Nada (nivel actual) |
| **Seasonal Naive** | El valor de hace un ciclo (`t − 52` semanas) | Estacionalidad |
| **Holt-Winters (ETS)** | Suavizado con nivel + tendencia + estacionalidad | Tendencia y estacionalidad |

Añadiremos un cuarto aún más tonto —la **media histórica**— para tener piso absoluto.

---
## 3. La partición temporal

Regla de oro del forecasting: **nunca evalúes con datos del pasado del entrenamiento**. Separamos el
**último año (52 semanas)** como *holdout* y entrenamos con todo lo anterior. El modelo pronostica esas 52
semanas "a ciegas" y comparamos contra lo que realmente pasó.

> Esto es una sola partición (*hold-out*), suficiente para comparar baselines. La evaluación robusta usa
> **validación cruzada temporal / walk-forward** — el tema del Video 20.

In [ ]:
H = 52                       # horizonte de prueba: un año
m = 52                       # período estacional (semanal → anual)
train, test = y.iloc[:-H], y.iloc[-H:]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(train.index, train.values, color=BLUE, linewidth=1.3, label=f'train ({len(train)} sem)')
ax.plot(test.index, test.values, color=GREEN, linewidth=1.8, label=f'holdout ({len(test)} sem)')
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Partición temporal: entrenar en el pasado, evaluar en el futuro',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()
print(f'Train: {train.index.min().date()} → {train.index.max().date()}')
print(f'Test : {test.index.min().date()} → {test.index.max().date()}')

---
## 4. Los baselines en acción

Generamos el pronóstico de las 52 semanas del holdout con cada método y los superponemos sobre lo real.

In [ ]:
mean_f = pd.Series(train.mean(), index=test.index)
naive = pd.Series(train.iloc[-1], index=test.index)
snaive = pd.Series(train.iloc[-m:].values[:H], index=test.index)
hw_add = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=m,
                              initialization_method='estimated').fit()
hw_mul = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=m,
                              initialization_method='estimated').fit()
hw_add_f = pd.Series(hw_add.forecast(H).values, index=test.index)
hw_mul_f = pd.Series(hw_mul.forecast(H).values, index=test.index)

forecasts = {
    'Media':          (mean_f,   '#94A3B8'),
    'Naive':          (naive,    ORANGE),
    'Seasonal Naive': (snaive,   GREEN),
    'Holt-Winters +': (hw_add_f, PURPLE),
    'Holt-Winters ×': (hw_mul_f, RED),
}

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index[-52:], train.values[-52:], color=BLUE, linewidth=1.2, alpha=0.6, label='train (último año)')
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL (holdout)')
for name, (f, c) in forecasts.items():
    ax.plot(test.index, f.values, color=c, linewidth=1.6, alpha=0.9, label=name)
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Pronóstico de cada baseline vs demanda real', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()
print('El Naive plano ignora la estacionalidad; el Seasonal Naive la replica casi calcada.')

---
## 5. Métricas y veredicto

Medimos con tres métricas complementarias (las formalizamos en el Video 19):

- **MAE** — error absoluto medio, en unidades.
- **WAPE** — error absoluto ponderado (`Σ|error| / Σ|real|`); robusto y en %, ideal cuando hay ceros.
- **MASE** — error escalado contra el *seasonal naive* en train: **< 1 le gana al baseline estacional;
  > 1 pierde**. Es la vara honesta.

In [ ]:
def mae(a, f):  return np.abs(a - f).mean()
def wape(a, f): return np.abs(a - f).sum() / np.abs(a).sum()

# Denominador MASE: MAE en train del seasonal naive (paso m)
mase_denom = np.abs(train.diff(m).dropna()).mean()

rows = []
for name, (f, _) in forecasts.items():
    rows.append({'modelo': name, 'MAE': mae(test, f), 'WAPE': wape(test, f),
                 'MASE': mae(test, f) / mase_denom})
report = pd.DataFrame(rows).set_index('modelo').sort_values('WAPE')

serios = report.drop('Naive')   # el Naive plano queda fuera de escala

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(serios.index[::-1], serios['WAPE'][::-1],
             color=[GREEN if v == serios['WAPE'].min() else BLUE for v in serios['WAPE'][::-1]])
axes[0].xaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title('WAPE por modelo (menor = mejor)', fontsize=12, fontweight='bold')

axes[1].barh(serios.index[::-1], serios['MASE'][::-1],
             color=[GREEN if v < 1 else RED for v in serios['MASE'][::-1]])
axes[1].axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='= seasonal naive')
axes[1].set_title('MASE (< 1 le gana al seasonal naive)', fontsize=12, fontweight='bold')
axes[1].legend()
fig.suptitle('Veredicto de los baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(report.assign(WAPE=lambda d: (d['WAPE'] * 100).round(1),
                    MAE=lambda d: d['MAE'].round(0),
                    MASE=lambda d: d['MASE'].round(2)).to_string())
ganador = report['WAPE'].idxmin()
print(f'\n→ Mejor baseline: {ganador} (WAPE {report.loc[ganador, "WAPE"]:.1%})')

---
## 6. La lección: lo simple que cuesta ganar

El resultado incomoda a quien cree que "más complejo = mejor":

- El **Naive plano** es un desastre (WAPE altísimo): ignorar la estacionalidad en una serie estacional es
  fatal. Sirve solo como piso.
- El **Seasonal Naive** —literalmente "copia el año pasado"— es **el mejor de todos aquí**. Con una serie
  de estacionalidad fuerte y estable, es un benchmark brutal.
- **Holt-Winters** es competitivo, pero **no le gana** al seasonal naive en esta serie: estimar 52 factores
  estacionales con solo 4 ciclos de historia lo penaliza.

**La consecuencia práctica:** cualquier modelo de la Fase 3 (SARIMA, Prophet, XGBoost) tendrá que
demostrar **MASE < 1** — es decir, ganarle a copiar el año pasado. Si no, no vale la pena.

---
## 7. Conclusiones y puente al Video 13

### Las reglas que te llevas

1. **Siempre establece un baseline antes de modelar.** Sin vara de medir, "bueno" no significa nada.
2. **El Seasonal Naive es el rival a vencer**, no el Naive plano. En series estacionales es sorprendentemente
   difícil de superar.
3. **Usa MASE como veredicto honesto:** < 1 le gana al seasonal naive; > 1 no aporta.
4. **Evalúa siempre en el futuro** (hold-out temporal), nunca sobre datos de entrenamiento. La versión
   robusta —walk-forward— llega en el Video 20.
5. **La complejidad se gana, no se asume.** Holt-Winters no le ganó a copiar el año pasado; tu modelo
   tampoco tiene derecho a hacerlo por defecto.

---

### Próximo video
**Video 13 — ARIMA, SARIMA y SARIMAX: el clásico que sí tienes que dominar**
Con el benchmark puesto, atacamos el modelo clásico: diferenciación (Video 3), lectura de ACF/PACF
(Video 5) y variables exógenas (precio y promo, Video 10) — y lo obligamos a superar el MASE < 1.